In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline

In [3]:
import torch
from nanodrz.model import DiarizeGPT, Config
from nanodrz.data import libritts_test, artificial_diarisation_sample
from nanodrz.utils import visualise_annotation, play
from nanodrz.download import dl_scp_file

ckpt = torch.load(dl_scp_file("gpudev:/home/harry/storj/runs/nanodrz/1706300953/0006000.pt"))

rsync --partial --progress --human-readable -e ssh gpudev:/home/harry/storj/runs/nanodrz/1706300953/0006000.pt /home/harry/storj/runs/nanodrz/1706300953/0006000.pt
0006000.pt

              0   0%    0.00kB/s    0:00:00  
        622.59K   0%  594.33kB/s    0:35:15  
          1.15M   0%  541.85kB/s    0:38:39  
          1.67M   0%  527.98kB/s    0:39:39  
          2.06M   0%  304.30kB/s    1:08:47  
          2.72M   0%  308.16kB/s    1:07:54  
          3.24M   0%  308.67kB/s    1:07:45  
          3.60M   0%  265.84kB/s    1:18:39  
          4.13M   0%  312.95kB/s    1:06:47  
          4.65M   0%  294.45kB/s    1:10:57  
          5.28M   0%  303.41kB/s    1:08:49  
          5.80M   0%  353.15kB/s    0:59:06  
          6.19M   0%  301.17kB/s    1:09:17  
          6.85M   0%  319.33kB/s    1:05:18  
          7.08M   0%  266.71kB/s    1:18:11  
          7.93M   0%  314.72kB/s    1:06:12  
          8.29M   0%  279.97kB/s    1:14:24  
          9.34M   0%  325.66kB/s    1:03:5

KeyboardInterrupt: 

In [ ]:
config = Config(**ckpt["config"])
model:DiarizeGPT = DiarizeGPT.from_pretrained(ckpt).cuda()

In [ ]:
print(config.model.use_time_pos)

In [ ]:
# Use the same parameters that the model was trained on to generate a sample
print(config.data.model_dump())
audio, labels = artificial_diarisation_sample(libritts_test(), **config.data.model_dump())
print(audio.shape[-1]/16000)
reference = visualise_annotation(labels)
play(audio)
labels
audio = audio.cuda()

In [ ]:
nlabels = model.generate(audio[None], temperature=.5, max_steps=(len(labels))*3)
print("\n".join([str(n) for n in nlabels]))
for l in nlabels:
    l[2] = l[2]+ "'"
hypothesis = visualise_annotation(labels+nlabels)

### Calculate Metrics with Pyannote


In [ ]:
!pip install pyannote.metrics

In [ ]:
from pyannote.core import Annotation
from pyannote.metrics.diarization import DiarizationErrorRate

metric = DiarizationErrorRate()
metric(reference, hypothesis)